# Train the groundwater models (Colab or Kaggle)

Local machines build the table; this notebook does the training, so heavier settings and longer searches are possible.

**What to upload:** `well_campaign_table.parquet` (about 40 MB, built by `python -m bits_ml.dataset`) and the `ml/bits_ml` package. Nothing else is needed: the table already carries the rain history, and its normals were computed from the training years only.

**What comes back:** a model bundle (`lgbm.joblib`) and a results table. Download both into `ml/models/` and `data/processed/` in the repo.

The split is fixed: train 2001–2014, tune on 2015–2017, and 2018–2022 stays untouched until the end.

In [ ]:
# 1. environment
# Pinned to the versions the repo runs locally: the trained model travels home as a pickle,
# and a different library version there can change how it behaves.
!pip -q install lightgbm==4.7.0 scikit-learn==1.9.1 pyarrow

import numpy as np, pandas as pd, lightgbm as lgb, sklearn, joblib
print('pandas', pd.__version__, '| lightgbm', lgb.__version__, '| sklearn', sklearn.__version__)

In [ ]:
# 2. the table and the package
#    Colab:  upload well_campaign_table.parquet and a zip of ml/bits_ml, or mount Drive
#    Kaggle: add them as a dataset and point TABLE at /kaggle/input/...
from pathlib import Path

TABLE = Path('well_campaign_table.parquet')
if not TABLE.exists():
    from google.colab import files  # noqa
    files.upload()

table = pd.read_parquet(TABLE)
for column in ('campaign', 'well_type', 'aquifer'):
    table[column] = table[column].astype('category')
print(f'{len(table):,} readings | {table.well_uid.nunique():,} wells | {table.year.min()}-{table.year.max()}')
table.groupby('campaign', observed=True)['anomaly_m'].agg(['size', 'std']).round(2)

In [ ]:
# 3. features, target and splits (kept identical to ml/bits_ml/dataset.py and splits.py)
WINDOWS = (3, 6, 12, 24, 36)
RAIN_FEATURES = ([f'rain_{w}m' for w in WINDOWS] + [f'rain_{w}m_pct' for w in WINDOWS]
                 + ['monsoon_last_mm', 'monsoon_last_mm_pct', 'monsoon_todate_mm',
                    'wet_days_12m', 'wet_days_12m_pct', 'max_day_12m'])
WELL_FEATURES = ['sy', 'well_depth_m', 'rain_normal_mm']
CATEGORICAL = ['campaign', 'well_type', 'aquifer']
FEATURES = RAIN_FEATURES + WELL_FEATURES + CATEGORICAL
TARGET = 'anomaly_m'

# More rain must never predict deeper water. Every rain input is constrained: a
# scenario raises all of them together, so leaving one free lets a wetter year
# come back as deeper water and breaks the what-if simulator.
MONOTONE = [-1 if f in RAIN_FEATURES else 0 for f in FEATURES]

TRAIN, VAL, TEST = range(2001, 2015), (2015, 2016, 2017), (2018, 2019, 2020, 2021, 2022)
rows = lambda years: table[table.year.isin(list(years))]
X = lambda frame: frame[FEATURES]
print({'train': len(rows(TRAIN)), 'val': len(rows(VAL)), 'test': len(rows(TEST))})

In [ ]:
# 4. baseline to beat: every well sits at its own normal
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def report(name, frame, pred):
    y = frame[TARGET].to_numpy()
    base = mean_squared_error(y, np.zeros_like(y))
    return {'model': name, 'n': len(y), 'rmse': np.sqrt(mean_squared_error(y, pred)),
            'mae': mean_absolute_error(y, pred), 'r2': r2_score(y, pred),
            'skill_vs_normal': 1 - mean_squared_error(y, pred) / base}

results = [report('baseline_normal', rows(VAL), np.zeros(len(rows(VAL))))]
pd.DataFrame(results).round(3)

In [ ]:
# 5. random search on the validation years (test years stay closed)
import itertools, random

space = {'num_leaves': [31, 63, 127], 'min_child_samples': [100, 200, 400], 'learning_rate': [0.03, 0.05],
         'colsample_bytree': [0.6, 0.8], 'reg_lambda': [1.0, 10.0]}
everything = [dict(zip(space, values)) for values in itertools.product(*space.values())]
random.seed(42)
combos = random.sample(everything, 24)  # 24 of 72: enough to find the shape, quick enough to sit through

best, trials = None, []
for i, params in enumerate(combos, 1):
    model = lgb.LGBMRegressor(objective='regression', n_estimators=3000, subsample=0.8, subsample_freq=1,
                              monotone_constraints=MONOTONE, monotone_constraints_method='advanced',
                              random_state=42, n_jobs=-1, verbose=-1, **params)
    model.fit(X(rows(TRAIN)), rows(TRAIN)[TARGET],
              eval_set=[(X(rows(VAL)), rows(VAL)[TARGET])],
              callbacks=[lgb.early_stopping(100, verbose=False)])
    score = report(' '.join(f'{k}={v}' for k, v in params.items()), rows(VAL), model.predict(X(rows(VAL))))
    score['best_iteration'] = model.best_iteration_
    trials.append(score)
    if best is None or score['rmse'] < best[0]['rmse']:
        best = (score, model)
    print(f"{i}/{len(combos)} rmse {score['rmse']:.4f} skill {score['skill_vs_normal']:+.4f}", flush=True)

pd.DataFrame(trials).sort_values('rmse').round(4).head(10)

In [ ]:
# 6. refit on train + validation with the chosen settings, then score the test years once
score, tuned = best
params = tuned.get_params() | {'n_estimators': max(200, int(tuned.best_iteration_ * 1.2))}
final = lgb.LGBMRegressor(**params).fit(X(rows(list(TRAIN) + list(VAL))), rows(list(TRAIN) + list(VAL))[TARGET])

test = rows(TEST)
final_scores = [report('lgbm tuned', test, final.predict(X(test))),
                report('baseline_normal', test, np.zeros(len(test)))]
per_campaign = [report(f'lgbm {c}', g, final.predict(X(g))) for c, g in test.groupby('campaign', observed=True)]
pd.DataFrame(final_scores + per_campaign).round(3)

In [ ]:
# 7. check the constraint holds: a wetter year must never come back as deeper water
probe = X(test).head(2000).copy()
before = final.predict(probe)
for column in RAIN_FEATURES:  # a scenario lifts every rain input together
    probe[column] = probe[column] * 1.2
after = final.predict(probe)
print('deeper anywhere with 20% more rain:', bool((after > before + 1e-9).any()))
print(f'median change: {np.median(after - before):+.3f} m (negative = water stands shallower)')

In [ ]:
# 8. save the bundle in the same shape ml/bits_ml/train.py writes, then download it
from datetime import datetime, timezone

# The category order is part of the model: LightGBM stores codes, not labels, so the repo
# has to put its table back into this exact order before predicting (dataset.apply_categories).
categories = {c: [str(v) for v in table[c].cat.categories] for c in CATEGORICAL}

bundle = {'model': final, 'name': 'lgbm', 'features': FEATURES, 'categories': categories, 'target': TARGET,
          'train_years': list(TRAIN) + list(VAL), 'n_rows': len(rows(list(TRAIN) + list(VAL))),
          'scores': {'test years': final_scores[0]}, 'search': trials,
          'built_at': datetime.now(timezone.utc).isoformat(timespec='seconds'),
          'versions': {'python': __import__('platform').python_version(),
                       'sklearn': sklearn.__version__, 'pandas': pd.__version__, 'lightgbm': lgb.__version__}}
joblib.dump(bundle, 'lgbm.joblib')
pd.DataFrame(final_scores + per_campaign).to_csv('model_results_colab.csv', index=False)

# a sample of predictions, so the repo can prove the model behaves identically after the trip home
check = test.head(2000)
pd.DataFrame({'well_uid': check.well_uid.to_numpy(), 'year': check.year.to_numpy(),
              'campaign': check.campaign.astype(str).to_numpy(),
              'prediction': final.predict(X(check))}).to_csv('colab_predictions.csv', index=False)
print('put lgbm.joblib in ml/models/, the two csv files in data/processed/')

try:
    from google.colab import files
    for name in ('lgbm.joblib', 'model_results_colab.csv', 'colab_predictions.csv'):
        files.download(name)
except Exception:
    pass  # Kaggle: the files are in the output panel